In [1]:
#hidden cell to be executed BEFORE the presentation
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import dftpy
from dftpy.ions import Ions
from dftpy.field import DirectField
from dftpy.grid import DirectGrid
from dftpy.functional import LocalPseudo, Functional, TotalFunctional
from dftpy.formats import io
from dftpy.math_utils import ecut2nr
from dftpy.time_data import TimeData
from dftpy.optimization import Optimization
from dftpy.mpi import sprint
from IPython.lib.display import YouTubeVideo
from IPython.display import IFrame
from ase.visualize import view
PP_list = {'Al': 'Al_lda.oe01.recpot'}
#import fortecubeview

<div id="bg-slide">
    <center>
    <h1>DFT introduction #1</h1>
<center>
<br>
<table>
  <tr>
      <td><p><h1>Team Rutgers</h1></p><p><h2>(Valeria, Ezekiel and Michele)</h2></p></td> 
      <td><img src="../figures/logos/ASESMA_logo.png" width=300 height=300 /></td>
  </tr>
  <tr>
    <td></td>
    <td>An interactive session</td>
  </tr>
</table>
<br>
<h3>Retrieve this presentation at:</h3>
<br>
<center>https://github.com/asesma-org/miniASESMA2026</center>

<br>

<center><h3> mini ASESMA 2026 $\bullet$ Accra, Ghana $\bullet$ June 16, 2026</h3></center>
</div>


# Goals of this lecture
- Reasons for considering DFT
- Basics of the theory behind DFT and Kohn-Sham DFT
- KS equations and XC functional

# Split into `N` groups

 - Assign number `1-N` to each student
 - Groups sit together
 - Possibly have 1 instructor per group

# The Real World
<table>
    <tr>
      <td><h3>Photocatalyst</h3></td>
        <td><h3>Catalytic nanoparticles</h3></td>
  </tr>
  <tr>
      <td><img src="../figures/science/photocatalyst.png" height=500 /></td>
      <td><img src="../figures/science/catalyst.png" height=500 /></td>
  </tr>
    <tr>
        <td>Chem. Comm., 43, 6551 (2009)</td>
        <td>PCCP, 21, 15080 (2019)</td>
    </tr>
</table>   

# Available electronic structure methods
<br>
<center>
    <img src="../figures/science/electronic_structure.png" width=1600 />
</center>

# Challenge 1

[Let's check your QM knowledge.](https://forms.office.com/r/VKyAnhNpxk) 
- What is the molecular Hamiltonian?

<center>
    <img src="../figures/random/qr_poll1.png" width=350 />
</center>


# Solution to Challenge #1 and a bit more...

Molecular Hamiltonian for interacting electrons and nuclei of charge $Z_\alpha$

$$
\hat{H} = \underbrace{-\frac{1}{2}\sum_i^{N_e} \nabla^2_i}_{\hat T} + \underbrace{\sum_i^{N_e}\sum_\alpha^{N_n} \frac{-Z_\alpha}{|r_i-R_\alpha|}}_{\hat V_{eN}} + \underbrace{\frac{1}{2}\sum_{i\neq j}^{N_e} \frac{1}{|r_i-r_j|}}_{\hat V_{ee}} + E_{NN}
$$

The Schrödinger equation for the ground state of the interacting system

$$
\hat H \Psi_0 = E_0 \Psi_0
$$

Expectation values of each term of the Hamiltonian

$$
T[\Psi_0] = \langle \Psi_0 | \hat T | \Psi_0 \rangle, ~~ E_{ee}[\Psi_0] = \langle \Psi_0 | \hat V_{ee} | \Psi_0 \rangle
$$

$$
E_{eN}[\Psi_0] = \color{red}{E_{eN}[n]} = \langle \Psi_0 | \hat V_{eN} | \Psi_0 \rangle = \int n(r) v_{eN}(r) dr
$$

# The Hohenberg and Kohn theorems
<br>
<br>
$$
\Psi_0 \longleftrightarrow n(r) \longleftrightarrow v_{eN}(r) \longleftrightarrow \Psi_0 
$$

Therefore $n(r)$, $v_{eN}(r)$ or $\Psi_0$ hold the same information. 

[HK paper 1964](../papers/PhysRev.136.B864.pdf)

In particular:

$$
E \equiv E[\Psi_0] \equiv E[v_{eN}] \equiv E[n]
$$

DFT exploits the latter as follows:

<center>
    <div class="alert alert-success"> 
    $$
     E[n] = T[n] + E_{ee}[n]+E_{eN}[n]+E_{NN}
    $$
    </div> 
</center>

# Challenge 2

[What do the energy functionals mean?](https://forms.office.com/r/74x4YMJ8Je) 
- Write an expression for each of the functionals ($T$, $E_{ee}$, $E_{eN}$) in terms of $n(r)$ and/or of $\Psi_0$.

<center>
    <img src="../figures/random/qr_poll1.2.png" width=350 />
</center>


# Solution to Challenge #2 and a bit more...

Recalling:
$$
\color{red}{E_{eN}[n]} = \langle \Psi_0 | \hat V_{eN} | \Psi_0 \rangle = \int n(r) v_{eN}(r) dr
$$
<center>
    <div class="alert alert-success"> 
    Because $v_{eN}$ is fixed by the system's geometry, $E_{eN}$ is already a density functional.
    </div> 
</center>

For the other terms in the Hamiltonian:
<br>
<br>
<br>

$$
T[\Psi_0] = \langle \Psi_0 | \hat T | \Psi_0 \rangle, ~~ E_{ee}[\Psi_0] = \langle \Psi_0 | \hat V_{ee} | \Psi_0 \rangle
$$
<br>
<br>

<center>
    <div class="alert alert-danger"> 
        Apparently, $T$ and $E_{ee}$ <b>cannot</b> be written as denisty functionals...
    </div> 
</center>


<h1> What can we do with the HK theorems? </h1>
<br>
<br>
<center>
<span style="font-size:45pt;"><i>               $n(r)$</i></span>
</center>
<br>
<br>
<br>
<center>...the density determines everything... in principle.... but is this useful? ... Are the functionals involved known?</center>

# Introducing: the Kohn-Sham (KS) system of *noninteracting* electrons
[KS paper 1965](../papers/PhysRev.140.A1133.pdf)

<center><img src="../figures/science/ks_system.png" width=1600 /></center>

<center> The <b>density</b> of the KS system is <b>the same</b> as the interacting system.</center>

<center style="font-size:20pt;"> $~~v_s(r) \longleftrightarrow n(r) \longleftrightarrow v_{eN}(r)$ </center>
<br>
<center> The <b>external potential</b> of the KS system is <b>different</b> from the interacting system.</center>

# Energy functional for the KS system

$$
E[n] = \color{red}{T[n] + E_{ee}[n]}+E_{eN}[n]+E_{NN}
$$

<center>
    <div class="alert alert-success"> 
        $$E[n] = \color{red}{T_s[n] + E_{H}[n] + E_{xc}[n]}+ E_{eN}[n]+ E_{NN}$$
        <br>
        where the density $n(r) = \sum_i^N |\phi_i(r)|^2$.
    </div> 
</center>

The KS kinetic energy is

$$
T_{s}[n] \equiv T_s[\{\phi_i\}]=  -\frac{1}{2}\sum_i \langle \phi_i | \nabla^2 | \phi_i\rangle = -\frac{1}{2}\sum_i  \int \phi_i^*(r) \nabla^2 \phi_i(r) dr
$$

<span style="color: red;">Mind: $T_s \neq T$.</span>

The classical e-e repulsion (Hartree) and e-N attraction:

$$
E_H[n]=\frac{1}{2}\int \frac{n(r)n(r')}{|r-r'|}drdr' \qquad\qquad E_{eN}[n] = \int n(r) v_{eN}(r) dr
$$

# Challenge 3: What is $E_{xc}$?


[Derive an expression for $E_{xc}$](https://forms.office.com/r/y0r5pAB5M4) 
- Derive an expression for $E_{xc}$ in terms of $T$, $E_{ee}$, $T_s$ and $E_H$.

<center>
    <img src="../figures/random/qr_poll3.png" width=350 />
</center>



# Solution to Challenge #3

The energy functional, whether KS or interacting, should yield the same energy value, therefore

$$
E_{xc}[n] = E_{ee}[n]-E_H[n] + T[n]-T_s[n]
$$


# Can $E_{xc}[n]$ be approximated? 

Usually it is approached by separating exchange, $E_x$, and correlation, $E_c$:
$$
E_{xc}[n]=E_{x}[n]+E_{c}[n]
$$

<center>
    <div class="alert alert-success">
        <b>Local Density Approximation</b> from the UEG. Example from Dirac's exchange:
        $$
        E_{x}^{\rm UEG} = - c_x \bar n^{4/3} V \longrightarrow E_x[n] \simeq -c_x \int n^{4/3}(r) dr
        $$
    </div>
</center>

 - <b>QE<span style="color: red;">py</span></b> uses same structure as QE to specify the `xc` functional

In [1]:
qe_options = {}
qe_options["&system"] = {}
qe_options["&system"]["input_dft"] = 'LDA'

 - Typically xc is already specified in the pseudopotential files.

# Solving for the electronic structure (find the KS orbitals, $\{\phi_i\}$)

To minimize the energy, we define an appropriate Lagrangian:

$$
\mathcal{L}_{KS}[\{\phi_i\}] = E[\{\phi_i\}] - \sum_{ij} \varepsilon_{ij}\left(\langle \phi_j|\phi_i \rangle - \delta_{ij}\right)
$$

At the minimum, we impose $\frac{\delta \mathcal{L}_{KS}[\{\phi_i\}]}{\delta \langle \phi_j|}=0$ or just $\frac{\delta \mathcal{L}_{KS}[\{\phi_i\}]}{\delta \phi_j^*(r)}=0$, and choose the so-called <b>canonical</b> orbitals (i.e., $\varepsilon_{ij}=\varepsilon_{i}\delta_{ij}$).

This yields the <span style="color: green;">Kohn-Sham equations</span>:

$$
-\frac{1}{2}\nabla^2 \phi_i(r) + v_s[n](r)\phi_i(r) = \varepsilon_i\phi_i(r)
$$

where $v_s(r)$ includes effects of e-N attraction and e-e repulsion. <span style="color: red;">But what is $v_s(r)$?</span>

# Challenge 4

Considering the chain rule of functional differentiation ($\phi^*$ and $\phi$ are considered independent variables):
$$
\frac{\delta F[n]}{\delta \phi_j^*(r)} = \int \frac{\delta F[n]}{\delta n(r')}\frac{\delta n(r')}{\delta \phi_j^*(r)}dr' = \frac{\delta F[n]}{\delta n(r)} \phi_j(r).
$$

<span style="color: green;">Show that the KS potential is given by:</span>

$$
\color{green}{v_s[n](r) = \frac{\delta E_{H}[n]}{\delta n(r)} + \frac{\delta E_{xc}[n]}{\delta n(r)} + v_{eN}(r)}
$$
$$
~~~~~~~~~~~~~~~~~~~~~~ \color{green}{= \int \frac{n(r')}{|r-r'|} dr' + \frac{\delta E_{xc}[n]}{\delta n(r)} + v_{eN}(r)}
$$

[Can you derive the KS equations?](https://forms.office.com/r/cwjAQaxRKq)

<center>
    <img src="../figures/random/qr_poll4.png" width=350 />
</center>


<br>
<br>
<center>
<span style="font-size:45pt;"><i>               Thank you!</i></span>
</center>
<br>
<br>